# Practical Assignment 4: Classification Models — Kaggle Dataset
**Student:** Paras Satpute  
**Division / Roll:** A1-05  
**PRN:** 202401040027

## Dataset
**Breast Cancer Wisconsin (Diagnostic)** — Kaggle listing: https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

The notebook uses the canonical Wisconsin Diagnostic Breast Cancer (WDBC) data distributed with scikit-learn, which is a packaged copy of the same UCI WDBC dataset mirrored on Kaggle. This avoids API credentials and makes the notebook reproducible in Colab. The dataset has 569 records, 30 numeric features, and a binary diagnosis target. Target encoding here is **1 = malignant, 0 = benign**.

> This is a separate dataset from the web-scraped company revenue dataset used in the first submission. It is an educational dataset, not a clinical diagnostic tool.

## Models
Only Decision Tree, KNN and Gaussian Naive Bayes are included. Each is implemented twice: using scikit-learn and manually from scratch.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, log_loss, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## 1. Load and understand the dataset

The source Kaggle dataset is a mirror of the UCI Wisconsin Diagnostic Breast Cancer data. The scikit-learn copy is used here for stable, no-credential execution. It contains 569 observations and 30 continuous numeric features. The original target coding is converted so malignant is the positive class.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
y = (dataset.target == 0).astype(int)  # 1 = malignant, 0 = benign
df = X.copy()
df['diagnosis'] = y.map({0:'Benign', 1:'Malignant'})
print('Shape:', df.shape)
print('Missing values:', int(df.isnull().sum().sum()))
print(df['diagnosis'].value_counts())
display(df.head())

In [ ]:
df.describe().T[['mean','std','min','max']].head(10)

## 2. Exploratory data analysis

In [ ]:
df['diagnosis'].value_counts().plot(kind='bar', title='Class Distribution')
plt.ylabel('Number of samples'); plt.xticks(rotation=0); plt.show()

plt.figure(figsize=(8,5))
plt.hist(X['mean radius'], bins=25)
plt.title('Distribution of Mean Radius'); plt.xlabel('Mean radius'); plt.ylabel('Frequency')
plt.show()

## 3. Train-test split and preprocessing

An 80:20 stratified split is used. Scaling is fitted only on the training partition. Decision Trees use the original feature scale; KNN and Gaussian Naive Bayes use standardized features. This avoids allowing test-set information to influence preprocessing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Train class balance:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Test class balance:', y_test.value_counts(normalize=True).round(3).to_dict())

## 4. From-scratch implementations

These implementations use NumPy and explicit mathematical operations. They do **not** call the corresponding scikit-learn estimator.

- **Decision Tree:** binary CART-style recursive splitting using Gini impurity. To keep runtime practical, at most 30 candidate thresholds per feature are tested.
- **KNN:** Euclidean distance, nearest-neighbour selection, and inverse-distance weighted voting.
- **Gaussian Naive Bayes:** class priors and per-feature Gaussian likelihoods combined in log space for numerical stability.

In [ ]:
def gini_impurity(y):
    if len(y) == 0:
        return 0.0
    proportions = np.bincount(np.asarray(y, dtype=int), minlength=2) / len(y)
    return 1.0 - np.sum(proportions ** 2)


class ScratchDecisionTree:
    def __init__(self, max_depth=4, min_samples_leaf=3, max_thresholds=30):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_thresholds = max_thresholds

    def fit(self, X, y):
        self.X = np.asarray(X, dtype=float)
        self.y = np.asarray(y, dtype=int)
        self.root = self._grow(np.arange(len(self.y)), depth=0)
        return self

    def _grow(self, indices, depth):
        labels = self.y[indices]
        node = {'prob': float(labels.mean())}
        if (depth >= self.max_depth or
            len(indices) < 2 * self.min_samples_leaf or
            len(np.unique(labels)) == 1):
            return node

        parent_impurity = gini_impurity(labels)
        best_gain, best_feature, best_threshold, best_mask = 0.0, None, None, None

        for feature in range(self.X.shape[1]):
            values = np.unique(self.X[indices, feature])
            if len(values) < 2:
                continue
            if len(values) > self.max_thresholds:
                q = np.linspace(0, 1, self.max_thresholds + 2)[1:-1]
                thresholds = np.unique(np.quantile(values, q))
            else:
                thresholds = (values[:-1] + values[1:]) / 2

            for threshold in thresholds:
                mask = self.X[indices, feature] <= threshold
                n_left, n_right = int(mask.sum()), int((~mask).sum())
                if min(n_left, n_right) < self.min_samples_leaf:
                    continue
                gain = (parent_impurity
                        - n_left / len(indices) * gini_impurity(self.y[indices[mask]])
                        - n_right / len(indices) * gini_impurity(self.y[indices[~mask]]))
                if gain > best_gain:
                    best_gain, best_feature = gain, feature
                    best_threshold, best_mask = float(threshold), mask

        if best_feature is None:
            return node
        node.update({
            'feature': best_feature, 'threshold': best_threshold,
            'left': self._grow(indices[best_mask], depth + 1),
            'right': self._grow(indices[~best_mask], depth + 1)
        })
        return node

    def _predict_one_probability(self, row, node):
        while 'feature' in node:
            if row[node['feature']] <= node['threshold']:
                node = node['left']
            else:
                node = node['right']
        return node['prob']

    def predict_proba(self, X):
        p = np.array([self._predict_one_probability(row, self.root)
                      for row in np.asarray(X, dtype=float)])
        return np.column_stack([1-p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class ScratchKNN:
    def __init__(self, n_neighbors=7, weights='distance'):
        self.n_neighbors = n_neighbors
        self.weights = weights

    def fit(self, X, y):
        self.X = np.asarray(X, dtype=float)
        self.y = np.asarray(y, dtype=int)
        return self

    def predict_proba(self, X):
        probabilities = []
        for row in np.asarray(X, dtype=float):
            distances = np.sqrt(np.sum((self.X - row) ** 2, axis=1))
            nearest = np.argsort(distances)[:self.n_neighbors]
            if self.weights == 'distance':
                weights = 1.0 / np.maximum(distances[nearest], 1e-12)
            else:
                weights = np.ones(len(nearest))
            p1 = np.sum(weights * self.y[nearest]) / np.sum(weights)
            probabilities.append(p1)
        p = np.asarray(probabilities)
        return np.column_stack([1-p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class ScratchGaussianNB:
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        self.priors = np.array([(y == c).mean() for c in (0, 1)])
        self.means = np.array([X[y == c].mean(axis=0) for c in (0, 1)])
        self.variances = np.array([X[y == c].var(axis=0) + 1e-9 for c in (0, 1)])
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        log_probabilities = []
        for c in (0, 1):
            log_likelihood = -0.5 * np.sum(
                np.log(2 * np.pi * self.variances[c])
                + (X - self.means[c]) ** 2 / self.variances[c], axis=1
            )
            log_probabilities.append(np.log(self.priors[c]) + log_likelihood)
        logp = np.vstack(log_probabilities).T
        logp -= logp.max(axis=1, keepdims=True)
        exp_logp = np.exp(logp)
        p = exp_logp[:, 1] / exp_logp.sum(axis=1)
        return np.column_stack([1-p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

## 5. Library implementations and training

The library versions use the corresponding Scikit-learn estimators. The model settings are deliberately kept comparable to the scratch versions: tree depth 4 and leaf minimum 3; KNN uses 7 neighbours with distance weighting; GaussianNB uses its standard Gaussian likelihood.

In [ ]:
library_models = {
    'Decision Tree': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=3,
                                             random_state=RANDOM_STATE),
                      X_train, X_test),
    'KNN': (KNeighborsClassifier(n_neighbors=7, weights='distance'),
            X_train_scaled, X_test_scaled),
    'Gaussian Naive Bayes': (GaussianNB(), X_train_scaled, X_test_scaled)
}
scratch_models = {
    'Decision Tree': (ScratchDecisionTree(max_depth=4, min_samples_leaf=3),
                      X_train, X_test),
    'KNN': (ScratchKNN(n_neighbors=7, weights='distance'),
            X_train_scaled, X_test_scaled),
    'Gaussian Naive Bayes': (ScratchGaussianNB(),
                             X_train_scaled, X_test_scaled)
}

## 6. Evaluation function

Accuracy, precision, recall and F1-score are calculated with malignant as the positive class. Log loss evaluates the quality of predicted probabilities; lower is better.

In [ ]:
results = []
predictions = {}
fitted_models = {}

def evaluate_models(model_dictionary, implementation_name):
    for model_name, (model, Xtr, Xte) in model_dictionary.items():
        model.fit(Xtr, y_train)
        prediction = model.predict(Xte)
        probability = model.predict_proba(Xte)[:, 1]
        row = {
            'Implementation': implementation_name,
            'Model': model_name,
            'Accuracy': accuracy_score(y_test, prediction),
            'Precision': precision_score(y_test, prediction, zero_division=0),
            'Recall': recall_score(y_test, prediction, zero_division=0),
            'F1-score': f1_score(y_test, prediction, zero_division=0),
            'Log loss': log_loss(y_test, probability)
        }
        results.append(row)
        predictions[(implementation_name, model_name)] = (prediction, probability)
        fitted_models[(implementation_name, model_name)] = model
        print(f'\\n{implementation_name} — {model_name}')
        print(pd.Series(row).to_string())
        print(classification_report(y_test, prediction,
                                    target_names=['Benign', 'Malignant']))

evaluate_models(library_models, 'Library')
evaluate_models(scratch_models, 'From scratch')
results_df = pd.DataFrame(results)
display(results_df.round(4))

## 7. Cross-validation for library models

Five-fold stratified cross-validation is performed on the training partition. The held-out test set remains untouched until final evaluation. CV scores are useful for judging stability, while the test scores estimate final generalization.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, (model, Xtr, _) in library_models.items():
    acc = cross_val_score(clone(model), Xtr, y_train, cv=cv, scoring='accuracy')
    f1 = cross_val_score(clone(model), Xtr, y_train, cv=cv, scoring='f1')
    cv_rows.append({'Model': name, 'CV accuracy mean': acc.mean(),
                    'CV accuracy std': acc.std(), 'CV F1 mean': f1.mean()})
cv_df = pd.DataFrame(cv_rows)
display(cv_df.round(4))

## 8. Visual comparison and confusion matrices

In [ ]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-score']
ax = results_df.set_index(['Model','Implementation'])[metric_cols].plot(
    kind='bar', figsize=(12,5), ylim=(0,1.05), rot=25)
ax.set_title('Library vs From-scratch model metrics')
ax.set_ylabel('Score'); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13,8))
for row_idx, implementation in enumerate(['Library','From scratch']):
    for col_idx, model_name in enumerate(['Decision Tree','KNN','Gaussian Naive Bayes']):
        pred = predictions[(implementation, model_name)][0]
        ConfusionMatrixDisplay.from_predictions(
            y_test, pred, display_labels=['Benign','Malignant'],
            ax=axes[row_idx,col_idx], colorbar=False)
        axes[row_idx,col_idx].set_title(f'{implementation}: {model_name}')
plt.tight_layout(); plt.show()

results_df.sort_values('F1-score', ascending=False)

## 9. Comparative analysis

Interpret results using the generated `results_df` rather than assuming one algorithm is always best.

- **Decision Tree:** easy to interpret and does not require scaling. The scratch implementation uses a restricted threshold search (up to 30 candidate cut points), so it is an educational CART approximation rather than a byte-for-byte replica of Scikit-learn's optimized tree.
- **KNN:** relies on distances, so scaling is essential. Scratch KNN explicitly calculates Euclidean distances and inverse-distance weighted votes. It has higher prediction cost because distances to training observations are calculated at inference time.
- **Gaussian Naive Bayes:** estimates a Gaussian distribution for each feature within each class and combines likelihoods with class priors. Its conditional-independence assumption is often unrealistic for correlated tumour measurements.
- Differences between scratch and library scores can arise from implementation details, numerical handling, tie-breaking and split-search strategy.

## 10. Ethics, limitations and conclusion

This dataset concerns medical diagnoses. The models are strictly educational and must not be used for diagnosis, screening or treatment decisions. The dataset is relatively small and historical; its population and collection process may not represent current clinical settings. Features and labels may contain measurement or sampling bias. False negatives (malignant cases classified as benign) are especially consequential, so recall and clinical validation matter in addition to accuracy.

**Conclusion:** Use the final metric table and confusion matrices to identify the strongest model on this split. No model should be described as clinically reliable. A larger independent dataset, external validation, calibration analysis and fairness assessment would be needed before any real-world use.

## Dataset citation
Kaggle: **Breast Cancer Wisconsin (Diagnostic) Data Set**  
https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data  
Underlying source: UCI Machine Learning Repository, Wisconsin Diagnostic Breast Cancer (WDBC).  
scikit-learn dataset documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

**Reproducibility:** random state 42; 80:20 stratified train-test split; 5-fold stratified CV on training data.